# 09 — LLM Output Evaluation

## Evaluation Framework

This notebook evaluates the full AI pipeline: **ML Model → SHAP → RAG → LLM** across 2,263 patients.

| Section | Method | Ground Truth |
|---|---|---|
| **1. Risk Level Alignment** | LLM risk_level vs ML risk_pct thresholds | ML prediction |
| **2. Pipeline vs Reality** | LLM HIGH/LOW vs actual readmitted_30d | ✅ Real labels |
| **3. Output Completeness** | All required fields present & non-empty | Structural check |
| **4. Doctor Alert Quality** | Urgency tone, length, SHAP driver mention | Reference-free |
| **5. Patient Precaution Readability** | Flesch reading ease, word count, jargon | Reference-free |
| **6. BERTScore** | Semantic similarity: doctor vs patient text | Embedding-based |
| **7. Summary Dashboard** | All metrics in one view | — |

In [ ]:
import json, os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import duckdb
from collections import Counter
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────
BASE_DIR  = os.path.normpath(os.path.join(os.path.abspath(''), '..'))
LLM_PATH  = os.path.join(BASE_DIR, 'llm_outputs_semantic.json')
DB_PATH   = os.path.join(BASE_DIR, 'dataset', 'hf_project.duckdb')

# ── Load LLM outputs ──────────────────────────────────────────
with open(LLM_PATH) as f:
    raw = json.load(f)
llm_records = [r for r in raw if 'error' not in r]
df_llm = pd.DataFrame(llm_records)
df_llm['hadm_id'] = df_llm['hadm_id'].astype(str)
df_llm['risk_level'] = df_llm['risk_level'].str.upper().str.strip()

# ── Load ground truth labels ───────────────────────────────────
con = duckdb.connect(DB_PATH, read_only=True)
df_labels = con.execute("SELECT hadm_id, readmitted_30d FROM model_features").fetchdf()
df_labels['hadm_id'] = df_labels['hadm_id'].astype(str)
con.close()

# ── Merge ─────────────────────────────────────────────────────
df = df_llm.merge(df_labels, on='hadm_id', how='inner')

print(f"LLM records loaded     : {len(df_llm):,}")
print(f"Matched with DB labels : {len(df):,}")
print(f"Actually readmitted    : {df['readmitted_30d'].sum():,} ({df['readmitted_30d'].mean()*100:.1f}%)")
print(f"\nRisk level distribution:")
print(df['risk_level'].value_counts().to_string())

# ── Plot style ─────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f12', 'axes.facecolor': '#0f0f12',
    'axes.edgecolor': '#333', 'text.color': '#f4f4f5',
    'axes.labelcolor': '#a1a1aa', 'xtick.color': '#71717a',
    'ytick.color': '#71717a', 'grid.color': '#1e1e24',
    'axes.titleweight': 'bold', 'axes.titlesize': 12,
    'font.family': 'DejaVu Sans'
})
COLORS = {'HIGH': '#ef4444', 'MEDIUM': '#f59e0b', 'LOW': '#10b981'}
print("\nSetup complete ✓")

## Section 1 — Risk Level Alignment
### Does the LLM's risk_level match the ML model's predicted_risk_pct?
- ML ≥ 60% → expected HIGH
- ML 30–60% → expected MEDIUM  
- ML < 30% → expected LOW

In [ ]:
def ml_to_tier(pct):
    if pct >= 60: return 'HIGH'
    if pct >= 30: return 'MEDIUM'
    return 'LOW'

df['expected_level'] = df['predicted_risk_pct'].apply(ml_to_tier)
df['level_aligned']  = df['risk_level'] == df['expected_level']

alignment_acc = df['level_aligned'].mean() * 100
print(f"Risk Level Alignment Accuracy: {alignment_acc:.1f}%\n")

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
labels = ['HIGH', 'MEDIUM', 'LOW']
cm = confusion_matrix(df['expected_level'], df['risk_level'], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix (rows=Expected from ML, cols=LLM assigned):")
print(cm_df.to_string())

# Per-tier breakdown
print("\nPer-tier alignment:")
for tier in labels:
    sub = df[df['expected_level'] == tier]
    acc = (sub['risk_level'] == tier).mean() * 100
    print(f"  {tier:8s}: {acc:.1f}% aligned  (n={len(sub):,})")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Section 1 — Risk Level Alignment', color='white', fontsize=14, fontweight='bold', y=1.01)

# Bar chart
tier_acc = [(t, (df[df['expected_level']==t]['risk_level']==t).mean()*100) for t in labels]
bars = axes[0].bar([t[0] for t in tier_acc], [t[1] for t in tier_acc],
                   color=[COLORS[t[0]] for t in tier_acc], width=0.5, edgecolor='none')
axes[0].axhline(alignment_acc, color='white', linestyle='--', alpha=0.5, label=f'Overall {alignment_acc:.1f}%')
axes[0].set_ylim(0, 110)
axes[0].set_title('Alignment Accuracy by Risk Tier', color='white')
axes[0].set_ylabel('% Aligned with ML Prediction')
axes[0].legend(facecolor='#1e1e24', labelcolor='white')
for bar, (_, val) in zip(bars, tier_acc):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', color='white', fontsize=11, fontweight='bold')

# Heatmap
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            cbar_kws={'shrink': 0.8}, linewidths=0.5,
            annot_kws={'color': 'white', 'fontsize': 12})
axes[1].set_title('Confusion Matrix\n(Expected vs LLM Assigned)', color='white')
axes[1].set_xlabel('LLM Risk Level', color='#a1a1aa')
axes[1].set_ylabel('Expected (from ML %)', color='#a1a1aa')
axes[1].tick_params(colors='#a1a1aa')

plt.tight_layout()
plt.savefig('eval_section1_alignment.png', dpi=150, bbox_inches='tight', facecolor='#0f0f12')
plt.show()
print(f"\n✅ Overall Alignment: {alignment_acc:.1f}%")

## Section 2 — Pipeline Evaluation vs Ground Truth (`readmitted_30d`)
### Does HIGH risk → actually readmitted? Precision, Recall, F1 for the full pipeline.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, precision_recall_curve

# Binary: HIGH = positive prediction
df['llm_binary']  = (df['risk_level'] == 'HIGH').astype(int)
df['ml_binary']   = (df['predicted_risk_pct'] >= 60).astype(int)
y_true = df['readmitted_30d']

print("=" * 55)
print("PIPELINE EVALUATION vs GROUND TRUTH (readmitted_30d)")
print("=" * 55)

# LLM-based classification
print("\n--- LLM Risk Level (HIGH = positive) ---")
print(classification_report(y_true, df['llm_binary'], target_names=['Not Readmitted', 'Readmitted']))

# ML-based classification
print("--- ML Score (≥60% = positive) ---")
print(classification_report(y_true, df['ml_binary'], target_names=['Not Readmitted', 'Readmitted']))

# AUC using continuous ML score
auc = roc_auc_score(y_true, df['predicted_risk_pct'] / 100)
print(f"ML Model AUC-ROC: {auc:.4f}")

# Readmission rate by LLM tier
print("\nReadmission rate by LLM risk tier:")
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    sub = df[df['risk_level'] == tier]
    rate = sub['readmitted_30d'].mean() * 100
    print(f"  {tier:8s}: {rate:.1f}%  (n={len(sub):,})")

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Section 2 — Pipeline Evaluation vs Ground Truth', color='white', fontsize=14, fontweight='bold')

# 1. Readmission rate by tier
tiers = ['HIGH', 'MEDIUM', 'LOW']
rates = [df[df['risk_level']==t]['readmitted_30d'].mean()*100 for t in tiers]
bars = axes[0].bar(tiers, rates, color=[COLORS[t] for t in tiers], width=0.5, edgecolor='none')
axes[0].set_title('Actual Readmission Rate\nby LLM Risk Tier', color='white')
axes[0].set_ylabel('% Actually Readmitted')
axes[0].set_ylim(0, 50)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{rate:.1f}%', ha='center', color='white', fontweight='bold')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_true, df['predicted_risk_pct'] / 100)
axes[1].plot(fpr, tpr, color='#3b82f6', lw=2, label=f'ML Model (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1], color='#444', linestyle='--', lw=1, label='Random (AUC=0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#3b82f6')
axes[1].set_title('ROC Curve — ML Pipeline', color='white')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(facecolor='#1e1e24', labelcolor='white', fontsize=9)

# 3. Risk score distribution by actual outcome
axes[2].hist(df[df['readmitted_30d']==1]['predicted_risk_pct'], bins=30,
             alpha=0.7, color='#ef4444', label='Readmitted (1)', edgecolor='none')
axes[2].hist(df[df['readmitted_30d']==0]['predicted_risk_pct'], bins=30,
             alpha=0.5, color='#3b82f6', label='Not Readmitted (0)', edgecolor='none')
axes[2].axvline(60, color='#ef4444', linestyle='--', alpha=0.8, lw=1, label='HIGH threshold (60%)')
axes[2].axvline(30, color='#f59e0b', linestyle='--', alpha=0.8, lw=1, label='MEDIUM threshold (30%)')
axes[2].set_title('Risk Score Distribution\nby Actual Outcome', color='white')
axes[2].set_xlabel('Predicted Risk %')
axes[2].set_ylabel('Patient Count')
axes[2].legend(facecolor='#1e1e24', labelcolor='white', fontsize=8)

plt.tight_layout()
plt.savefig('eval_section2_ground_truth.png', dpi=150, bbox_inches='tight', facecolor='#0f0f12')
plt.show()

## Section 3 — Output Completeness & Quality
### Are all required fields present and non-empty?

In [ ]:
def get_risk_summary(row):
    da = row.get('doctor_alert', {})
    if isinstance(da, dict): return da.get('risk_summary', '')
    return ''

def get_precautions(row):
    p = row.get('patient_precautions', [])
    return p if isinstance(p, list) else []

def get_followup(row):
    f = row.get('follow_up_recommendations', '')
    return f if isinstance(f, str) else ''

records = llm_records

# Completeness checks
has_risk_level    = [bool(r.get('risk_level','').strip()) for r in records]
has_risk_summary  = [bool(get_risk_summary(r).strip()) for r in records]
has_precautions   = [len(get_precautions(r)) >= 4 for r in records]
has_followup      = [bool(get_followup(r).strip()) for r in records]
all_complete      = [a and b and c and d for a,b,c,d in
                     zip(has_risk_level, has_risk_summary, has_precautions, has_followup)]

metrics = {
    'Risk Level Present':       sum(has_risk_level),
    'Doctor Alert Filled':      sum(has_risk_summary),
    '4+ Patient Precautions':   sum(has_precautions),
    'Follow-up Present':        sum(has_followup),
    'Fully Complete':           sum(all_complete),
}

total = len(records)
print(f"Total LLM records: {total:,}\n")
print(f"{'Field':<30} {'Count':>8} {'%':>8}")
print("-" * 48)
for k, v in metrics.items():
    status = '✅' if v/total >= 0.95 else '⚠️' if v/total >= 0.80 else '❌'
    print(f"{k:<30} {v:>8,} {v/total*100:>7.1f}%  {status}")

# Lengths
summary_lens   = [len(get_risk_summary(r).split()) for r in records if get_risk_summary(r)]
prec_lens      = [len(' '.join(get_precautions(r)).split()) / max(len(get_precautions(r)),1)
                  for r in records if get_precautions(r)]
followup_lens  = [len(get_followup(r).split()) for r in records if get_followup(r)]

print(f"\nText length stats (words):")
print(f"  Doctor alert summary : avg={np.mean(summary_lens):.0f}  min={min(summary_lens)}  max={max(summary_lens)}")
print(f"  Per precaution       : avg={np.mean(prec_lens):.0f}  min={int(min(prec_lens))}  max={int(max(prec_lens))}")
print(f"  Follow-up recs       : avg={np.mean(followup_lens):.0f}  min={min(followup_lens)}  max={max(followup_lens)}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Section 3 — Output Completeness & Quality', color='white', fontsize=14, fontweight='bold')

fields = list(metrics.keys())
values = [v/total*100 for v in metrics.values()]
colors = ['#10b981' if v >= 95 else '#f59e0b' if v >= 80 else '#ef4444' for v in values]
bars = axes[0].barh(fields, values, color=colors, edgecolor='none')
axes[0].axvline(95, color='white', linestyle='--', alpha=0.4, label='95% threshold')
axes[0].set_xlim(0, 105)
axes[0].set_title('Field Completeness Rate', color='white')
axes[0].set_xlabel('% of Records')
axes[0].legend(facecolor='#1e1e24', labelcolor='white')
for bar, val in zip(bars, values):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', color='white', fontsize=10)

# Length distributions
axes[1].hist(summary_lens, bins=30, color='#3b82f6', alpha=0.8, edgecolor='none', label='Doctor Alert')
axes[1].hist(followup_lens, bins=30, color='#10b981', alpha=0.7, edgecolor='none', label='Follow-up')
axes[1].set_title('Output Text Length Distribution\n(word count)', color='white')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Patient Count')
axes[1].legend(facecolor='#1e1e24', labelcolor='white')

plt.tight_layout()
plt.savefig('eval_section3_completeness.png', dpi=150, bbox_inches='tight', facecolor='#0f0f12')
plt.show()

## Section 4 — Doctor Alert Quality (Reference-Free)
### Urgency tone analysis: do HIGH risk alerts use more urgent clinical language?

In [ ]:
URGENT_WORDS   = ['urgent', 'immediately', 'critical', 'severe', 'aggressive',
                  'closely', 'high risk', 'emergency', 'serious', 'significant']
CLINICAL_TERMS = ['bicarbonate', 'creatinine', 'hemoglobin', 'sodium', 'potassium',
                  'ejection fraction', 'bun', 'troponin', 'diuretic', 'shap',
                  'readmission', 'polypharmacy', 'electrolyte', 'renal']

def count_keywords(text, keywords):
    text_l = text.lower()
    return sum(1 for w in keywords if w in text_l)

df['alert_text']      = df.apply(lambda r: get_risk_summary(r.to_dict()), axis=1)
df['urgency_score']   = df['alert_text'].apply(lambda t: count_keywords(t, URGENT_WORDS))
df['clinical_score']  = df['alert_text'].apply(lambda t: count_keywords(t, CLINICAL_TERMS))
df['alert_len_words'] = df['alert_text'].apply(lambda t: len(t.split()))

print("Doctor Alert Quality by Risk Tier")
print("="*65)
print(f"{'Tier':<10} {'n':>6} {'Urgency Score':>14} {'Clinical Terms':>15} {'Avg Words':>10}")
print("-"*65)
for tier in ['HIGH', 'MEDIUM', 'LOW']:
    sub = df[df['risk_level'] == tier]
    print(f"{tier:<10} {len(sub):>6,} {sub['urgency_score'].mean():>14.2f} "
          f"{sub['clinical_score'].mean():>15.2f} {sub['alert_len_words'].mean():>10.1f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Section 4 — Doctor Alert Quality (Reference-Free)', color='white', fontsize=14, fontweight='bold')

tiers = ['HIGH', 'MEDIUM', 'LOW']

# Urgency score by tier
urgency_means = [df[df['risk_level']==t]['urgency_score'].mean() for t in tiers]
axes[0].bar(tiers, urgency_means, color=[COLORS[t] for t in tiers], width=0.5, edgecolor='none')
axes[0].set_title('Urgency Keyword Score\nby Risk Tier', color='white')
axes[0].set_ylabel('Avg Urgency Keywords per Alert')
for i, v in enumerate(urgency_means):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', color='white', fontweight='bold')

# Clinical term score
clin_means = [df[df['risk_level']==t]['clinical_score'].mean() for t in tiers]
axes[1].bar(tiers, clin_means, color=[COLORS[t] for t in tiers], width=0.5, edgecolor='none')
axes[1].set_title('Clinical Term Richness\nby Risk Tier', color='white')
axes[1].set_ylabel('Avg Clinical Terms per Alert')
for i, v in enumerate(clin_means):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', color='white', fontweight='bold')

# Alert length distribution
for tier in tiers:
    sub = df[df['risk_level'] == tier]
    axes[2].hist(sub['alert_len_words'], bins=20, alpha=0.6, color=COLORS[tier],
                 edgecolor='none', label=f'{tier} (n={len(sub)})')
axes[2].set_title('Alert Length Distribution\n(words)', color='white')
axes[2].set_xlabel('Word Count')
axes[2].set_ylabel('Count')
axes[2].legend(facecolor='#1e1e24', labelcolor='white')

plt.tight_layout()
plt.savefig('eval_section4_doctor_alert.png', dpi=150, bbox_inches='tight', facecolor='#0f0f12')
plt.show()

## Section 5 — Patient Precaution Readability
### Are patient precautions written in simple, understandable language?
Uses Flesch Reading Ease score: 0=very hard, 100=very easy. Target: ≥ 60 (plain English).

In [ ]:
def syllable_count(word):
    word = word.lower().strip(".,!?;:")
    count = max(1, len(re.findall(r'[aeiou]+', word)))
    if word.endswith('e') and len(word) > 2:
        count = max(1, count - 1)
    return count

def flesch_reading_ease(text):
    """Flesch Reading Ease: higher = easier to read."""
    sentences = max(1, len(re.split(r'[.!?]', text)))
    words     = text.split()
    if not words: return 0
    syllables = sum(syllable_count(w) for w in words)
    return 206.835 - 1.015*(len(words)/sentences) - 84.6*(syllables/len(words))

JARGON = ['hemoglobin', 'creatinine', 'bicarbonate', 'electrolyte', 'diuretic',
          'ejection fraction', 'cardiomyopathy', 'troponin', 'pharmacology',
          'contraindicated', 'titrate', 'inotropic', 'arrhythmia']

def jargon_count(text):
    return sum(1 for j in JARGON if j in text.lower())

all_precs = []
for r in llm_records:
    tier  = r.get('risk_level', '').upper().strip()
    precs = get_precautions(r)
    combined = ' '.join(precs)
    if combined.strip():
        all_precs.append({
            'tier':      tier,
            'text':      combined,
            'flesch':    flesch_reading_ease(combined),
            'jargon':    jargon_count(combined),
            'word_count': len(combined.split()),
            'n_precs':   len(precs)
        })

df_prec = pd.DataFrame(all_precs)
df_prec = df_prec[df_prec['tier'].isin(['HIGH','MEDIUM','LOW'])]

print("Patient Precaution Readability by Risk Tier")
print("="*70)
print(f"{'Tier':<10} {'n':>6} {'Flesch Score':>13} {'Jargon Count':>13} {'Avg Words':>10}")
print("-"*70)
for tier in ['HIGH','MEDIUM','LOW']:
    sub = df_prec[df_prec['tier']==tier]
    print(f"{tier:<10} {len(sub):>6,} {sub['flesch'].mean():>13.1f} "
          f"{sub['jargon'].mean():>13.2f} {sub['word_count'].mean():>10.1f}")

pct_easy = (df_prec['flesch'] >= 60).mean() * 100
print(f"\n% of records with Flesch ≥ 60 (plain English): {pct_easy:.1f}%")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Section 5 — Patient Precaution Readability', color='white', fontsize=14, fontweight='bold')

# Flesch score distribution
for tier in ['HIGH','MEDIUM','LOW']:
    sub = df_prec[df_prec['tier']==tier]
    axes[0].hist(sub['flesch'], bins=25, alpha=0.65, color=COLORS[tier],
                 edgecolor='none', label=f'{tier}')
axes[0].axvline(60, color='white', linestyle='--', alpha=0.7, label='Target ≥ 60')
axes[0].set_title('Flesch Reading Ease\nDistribution', color='white')
axes[0].set_xlabel('Score (higher = easier)')
axes[0].set_ylabel('Count')
axes[0].legend(facecolor='#1e1e24', labelcolor='white', fontsize=9)

# Avg Flesch by tier
flesch_means = [df_prec[df_prec['tier']==t]['flesch'].mean() for t in ['HIGH','MEDIUM','LOW']]
bars = axes[1].bar(['HIGH','MEDIUM','LOW'], flesch_means,
                   color=[COLORS[t] for t in ['HIGH','MEDIUM','LOW']], width=0.5, edgecolor='none')
axes[1].axhline(60, color='white', linestyle='--', alpha=0.5, label='Target 60')
axes[1].set_ylim(0, 90)
axes[1].set_title('Avg Flesch Score by Tier', color='white')
axes[1].set_ylabel('Flesch Reading Ease')
axes[1].legend(facecolor='#1e1e24', labelcolor='white')
for bar, val in zip(bars, flesch_means):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val:.1f}', ha='center', color='white', fontweight='bold')

# Jargon count by tier
jargon_means = [df_prec[df_prec['tier']==t]['jargon'].mean() for t in ['HIGH','MEDIUM','LOW']]
bars2 = axes[2].bar(['HIGH','MEDIUM','LOW'], jargon_means,
                    color=[COLORS[t] for t in ['HIGH','MEDIUM','LOW']], width=0.5, edgecolor='none')
axes[2].set_title('Medical Jargon Usage\nby Tier (lower = better for patients)', color='white')
axes[2].set_ylabel('Avg Jargon Terms per Record')
for bar, val in zip(bars2, jargon_means):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.2f}', ha='center', color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('eval_section5_readability.png', dpi=150, bbox_inches='tight', facecolor='#0f0f12')
plt.show()

## Section 6 — BERTScore
### Semantic similarity between doctor alert summaries and patient precautions.
High score = they cover the same topics. Too high (≈1.0) = identical = bad differentiation.

In [ ]:
from bert_score import score as bert_score

# Sample 200 patients (BERTScore is slow on full dataset)
SAMPLE_N = 200
sample = [r for r in llm_records
          if get_risk_summary(r) and get_precautions(r)]
sample = sample[:SAMPLE_N]

doctor_texts  = [get_risk_summary(r) for r in sample]
patient_texts = [' '.join(get_precautions(r)) for r in sample]
tiers_sample  = [r.get('risk_level','').upper() for r in sample]

print(f"Running BERTScore on {len(sample)} samples...")
P, R, F1 = bert_score(patient_texts, doctor_texts, lang='en', verbose=False)
f1_scores = F1.numpy()

print(f"\nBERTScore Results (Patient Precautions vs Doctor Alert)")
print(f"  Mean F1  : {f1_scores.mean():.4f}")
print(f"  Std F1   : {f1_scores.std():.4f}")
print(f"  Min F1   : {f1_scores.min():.4f}")
print(f"  Max F1   : {f1_scores.max():.4f}")

# By tier
print(f"\nBERTScore F1 by Risk Tier:")
for tier in ['HIGH','MEDIUM','LOW']:
    idx = [i for i,t in enumerate(tiers_sample) if t == tier]
    if idx:
        tier_scores = f1_scores[idx]
        print(f"  {tier:8s}: {tier_scores.mean():.4f} ± {tier_scores.std():.4f}  (n={len(idx)})")

print(f"\nInterpretation:")
mean_f1 = f1_scores.mean()
if mean_f1 > 0.90:
    print("  ⚠️  F1 > 0.90 — texts are very similar, limited differentiation between doctor/patient")
elif mean_f1 > 0.80:
    print("  ✅ F1 0.80–0.90 — good: same clinical topics but different language registers")
else:
    print("  ℹ️  F1 < 0.80 — texts diverge significantly in content")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Section 6 — BERTScore (Semantic Similarity)', color='white', fontsize=14, fontweight='bold')

# Overall distribution
axes[0].hist(f1_scores, bins=30, color='#3b82f6', alpha=0.85, edgecolor='none')
axes[0].axvline(f1_scores.mean(), color='white', linestyle='--', lw=2,
                label=f'Mean = {f1_scores.mean():.3f}')
axes[0].axvline(0.85, color='#10b981', linestyle=':', lw=1.5, label='Good threshold (0.85)')
axes[0].set_title('BERTScore F1 Distribution\n(Patient vs Doctor Text)', color='white')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Count')
axes[0].legend(facecolor='#1e1e24', labelcolor='white')

# By tier
tier_means = []
tier_stds  = []
valid_tiers = []
for tier in ['HIGH','MEDIUM','LOW']:
    idx = [i for i,t in enumerate(tiers_sample) if t == tier]
    if idx:
        tier_means.append(f1_scores[idx].mean())
        tier_stds.append(f1_scores[idx].std())
        valid_tiers.append(tier)

bars = axes[1].bar(valid_tiers, tier_means,
                   color=[COLORS[t] for t in valid_tiers],
                   yerr=tier_stds, capsize=5, width=0.5, edgecolor='none',
                   error_kw={'ecolor':'white','alpha':0.5})
axes[1].set_ylim(0.7, 1.0)
axes[1].axhline(0.85, color='#10b981', linestyle=':', lw=1.5, label='Good threshold (0.85)')
axes[1].set_title('BERTScore F1 by Risk Tier\n(mean ± std)', color='white')
axes[1].set_ylabel('F1 Score')
axes[1].legend(facecolor='#1e1e24', labelcolor='white')
for bar, val in zip(bars, tier_means):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f'{val:.3f}', ha='center', color='white', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('eval_section6_bertscore.png', dpi=150, bbox_inches='tight', facecolor='#0f0f12')
plt.show()

## Section 7 — Final Summary Dashboard
### All evaluation metrics in one consolidated view.

In [ ]:
import json, duckdb, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ── Reload base data if this cell is run standalone ──────────────────────────
if 'df' not in dir():
    import pandas as pd
    BASE = os.path.dirname(os.path.abspath("09_LLM_Evaluation.ipynb")) if "__file__" not in dir() else os.path.dirname(__file__)
    LLM_PATH = os.path.join(BASE, "..", "data", "llm_outputs_semantic.json")
    DB_PATH  = os.path.join(BASE, "..", "data", "mimic_hf.duckdb")
    with open(LLM_PATH) as _f:
        raw = json.load(_f)
    rows = []
    for hadm_id, rec in (raw.items() if isinstance(raw, dict) else [(r.get("hadm_id"), r) for r in raw]):
        rl = (rec.get("doctor_alert") or {}).get("risk_level", rec.get("risk_level", "UNKNOWN"))
        rows.append({"hadm_id": int(hadm_id), "risk_level": rl})
    df_llm = pd.DataFrame(rows)
    con = duckdb.connect(DB_PATH)
    df_gt = con.execute("SELECT hadm_id, readmitted_30d FROM model_features").fetchdf()
    df = df_llm.merge(df_gt, on="hadm_id", how="inner")
    df["llm_binary"] = (df["risk_level"].str.upper() == "HIGH").astype(int)

# ── Safe fallbacks for optional earlier-section variables ────────────────────
_alignment_acc = alignment_acc if "alignment_acc" in dir() else None
_auc           = auc           if "auc"           in dir() else None
_all_complete  = all_complete  if "all_complete"  in dir() else None
_flesch_mean   = df_prec["flesch"].mean() if "df_prec" in dir() and len(df_prec) > 0 else None
_pct_easy      = pct_easy      if "pct_easy"      in dir() else None
_bert_f1       = f1_scores.mean() if "f1_scores"  in dir() else None

# Compute what we can from df directly
y_true_arr = df["readmitted_30d"].values
y_pred_arr = df["llm_binary"].values

precision = precision_score(y_true_arr, y_pred_arr, zero_division=0)
recall    = recall_score(y_true_arr, y_pred_arr, zero_division=0)
f1        = f1_score(y_true_arr, y_pred_arr, zero_division=0)

if _auc is None:
    try:
        _auc = roc_auc_score(y_true_arr, y_pred_arr)
    except Exception:
        _auc = 0.0

if _alignment_acc is None:
    # Compute alignment: HIGH→1 matches readmitted=1, LOW/MEDIUM→0 matches readmitted=0
    aligned = ((df["risk_level"].str.upper() == "HIGH") == (df["readmitted_30d"] == 1))
    _alignment_acc = aligned.mean() * 100

if _all_complete is None:
    _all_complete = [True] * len(df)  # can't check without raw outputs here

def _fmt(v, is_pct=False, decimals=4):
    if v is None:
        return "N/A"
    return f"{v:.1f}%" if is_pct else f"{v:.{decimals}f}"

summary = {
    "Risk Level Alignment":        _fmt(_alignment_acc, is_pct=True),
    "AUC-ROC (ML Pipeline)":       _fmt(_auc),
    "Precision (HIGH=readmitted)": _fmt(precision),
    "Recall (HIGH=readmitted)":    _fmt(recall),
    "F1 Score (Pipeline)":         _fmt(f1),
    "Output Completeness":         _fmt(sum(_all_complete)/len(_all_complete)*100 if _all_complete else None, is_pct=True),
    "Avg Flesch Score (Patient)":  _fmt(_flesch_mean, decimals=1),
    "% Plain English (>=60)":      _fmt(_pct_easy, is_pct=True),
    "BERTScore F1 Mean":           _fmt(_bert_f1),
}

print("=" * 55)
print("   LLM EVALUATION SUMMARY — HEALTHCARE AI PIPELINE")
print("=" * 55)
for metric, value in summary.items():
    status = ""
    if value == "N/A":
        status = "⚪ N/A"
    elif "%" in value:
        v = float(value.replace("%",""))
        status = "✅" if v >= 80 else "⚠️" if v >= 60 else "❌"
    else:
        v = float(value)
        if metric == "AUC-ROC (ML Pipeline)":
            status = "✅" if v >= 0.65 else "⚠️"
        elif "Flesch" in metric:
            status = "✅" if v >= 60 else "⚠️"
        elif "BERTScore" in metric:
            status = "✅" if 0.75 <= v <= 0.92 else "⚠️"
        else:
            status = "✅" if v >= 0.3 else "⚠️"
    print(f"  {metric:<35} {value:>10}  {status}")
print("=" * 55)

# ── Summary table chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#0f0f12")
ax.set_facecolor("#0f0f12")
ax.axis("off")

col_labels = ["Metric", "Value", "Status"]
row_data = []
for metric, value in summary.items():
    if value == "N/A":
        status = "⚪ N/A"
    elif "%" in value:
        v = float(value.replace("%",""))
        status = "✅ PASS" if v >= 80 else "⚠️ FAIR" if v >= 60 else "❌ FAIL"
    else:
        v = float(value)
        if metric == "AUC-ROC (ML Pipeline)":
            status = "✅ PASS" if v >= 0.65 else "⚠️ FAIR"
        elif "Flesch" in metric:
            status = "✅ PASS" if v >= 60 else "⚠️ FAIR"
        elif "BERTScore" in metric:
            status = "✅ PASS" if 0.75 <= v <= 0.92 else "⚠️ FAIR"
        else:
            status = "✅ PASS" if v >= 0.3 else "⚠️ FAIR"
    row_data.append([metric, value, status])

table = ax.table(
    cellText=row_data,
    colLabels=col_labels,
    cellLoc="left",
    loc="center",
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(11)

for (row, col), cell in table.get_celld().items():
    cell.set_facecolor("#1a1a24" if row % 2 == 0 else "#0f0f12")
    cell.set_text_props(color="#f4f4f5")
    cell.set_edgecolor("#333")
    if row == 0:
        cell.set_facecolor("#1e3a5f")
        cell.set_text_props(color="white", fontweight="bold")
    if col == 2 and row > 0:
        text = cell.get_text().get_text()
        if "PASS" in text: cell.set_facecolor("#0d3320")
        elif "FAIR" in text: cell.set_facecolor("#3d2f00")
        elif "FAIL" in text: cell.set_facecolor("#3d0d0d")

ax.set_title("LLM Evaluation Summary — Healthcare Decision Intelligence Agent",
             color="white", fontsize=13, fontweight="bold", pad=10)

plt.tight_layout()
out_png = os.path.join(os.path.dirname(os.path.abspath("09_LLM_Evaluation.ipynb")) if "__file__" not in dir() else os.path.dirname(__file__), "eval_section7_summary.png")
plt.savefig(out_png, dpi=150, bbox_inches="tight", facecolor="#0f0f12")
plt.show()
print(f"\n✅ Evaluation complete. Summary saved → eval_section7_summary.png")
